In [1]:
import os
import pandas as pd
from collections import defaultdict
from datetime import datetime
import json
import logger

data_path = "assets/初赛数据/"
tmp_path = "tmp_data"
output_path = "data"

os.makedirs(tmp_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

In [2]:
# 合并数据


def merge_csv_files(input_path, out_path):
    # 根据前缀分组文件
    file_groups = defaultdict(list)
    for file_name in os.listdir(input_path):
        if file_name.endswith(".csv") and "字段释义" not in file_name:
            prefix = file_name.rsplit("_", 1)[0]
            file_groups[prefix].append(os.path.join(input_path, file_name))
    # 合并前缀相同的文件
    for prefix, file_list in file_groups.items():
        merged_df = pd.concat(
            (pd.read_csv(file) for file in file_list), ignore_index=True
        )
        output_file = os.path.join(out_path, f"{prefix}.csv")
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        merged_df.to_csv(output_file, index=False)
        logger.info(f'合并前缀为"{prefix}"的文件到{output_file}')
    # 将设备参数详情表转为csv
    df_device = pd.read_excel(f"{data_path}设备参数详情.xlsx")
    df_device.to_csv(os.path.join(out_path, "设备参数详情表.csv"), index=False)

logger.special("开始合并数据")
merge_csv_files(data_path, tmp_path)
merge_csv_files(data_path, output_path)
logger.success("数据合并完成")

2025-02-21 23:13:42.859 [SPECIAL] 开始合并数据
2025-02-21 23:13:43.117 [INFO] 合并前缀为"Ajia_plc_1"的文件到tmp_data\Ajia_plc_1.csv
2025-02-21 23:13:43.519 [INFO] 合并前缀为"device_13_11_meter_1311"的文件到tmp_data\device_13_11_meter_1311.csv
2025-02-21 23:13:44.273 [INFO] 合并前缀为"device_13_14_meter_1314"的文件到tmp_data\device_13_14_meter_1314.csv
2025-02-21 23:13:44.862 [INFO] 合并前缀为"device_13_2_meter_1302"的文件到tmp_data\device_13_2_meter_1302.csv
2025-02-21 23:13:45.280 [INFO] 合并前缀为"device_13_3_meter_1303"的文件到tmp_data\device_13_3_meter_1303.csv
2025-02-21 23:13:45.770 [INFO] 合并前缀为"device_1_15_meter_115"的文件到tmp_data\device_1_15_meter_115.csv
2025-02-21 23:13:46.152 [INFO] 合并前缀为"device_1_2_meter_102"的文件到tmp_data\device_1_2_meter_102.csv
2025-02-21 23:13:46.535 [INFO] 合并前缀为"device_1_3_meter_103"的文件到tmp_data\device_1_3_meter_103.csv
2025-02-21 23:13:46.920 [INFO] 合并前缀为"device_1_5_meter_105"的文件到tmp_data\device_1_5_meter_105.csv
2025-02-21 23:13:47.233 [INFO] 合并前缀为"Jiaoche_plc_1"的文件到tmp_data\Jiaoche_plc_1.csv
2025-02-21 

In [3]:
# 判定A架的开关机和有无电流


def convert_to_numeric(value):
    """
    将值转换为数值类型，无法转换的返回 -1
    """
    try:
        return float(value)
    except ValueError:
        return -1


logger.special("开始判定A架开关机和有无电流")
df = pd.read_csv(os.path.join(tmp_path, "Ajia_plc_1.csv"))
df["Ajia-3_v"] = df["Ajia-3_v"].apply(convert_to_numeric)
df["Ajia-5_v"] = df["Ajia-5_v"].apply(convert_to_numeric)
df["status"] = "False"
df["check_current_presence"] = "False"

for i in range(1, df.shape[0]):
    prev_ajia3 = df.loc[i - 1, "Ajia-3_v"]
    prev_ajia5 = df.loc[i - 1, "Ajia-5_v"]
    curr_ajia3 = df.loc[i, "Ajia-3_v"]
    curr_ajia5 = df.loc[i, "Ajia-5_v"]

    # 停电条件：当前 Ajia-5_v == -1，且前一时刻 Ajia-5_v > 0 或 0
    if curr_ajia5 == -1 and (prev_ajia5 >= 0):
        df.loc[i, "status"] = "停电"

    # A架开机条件：前一时刻 Ajia-3_v == -1，且当前 Ajia-3_v >= 0
    if prev_ajia3 == -1 and curr_ajia3 >= 0:
        df.loc[i, "status"] = "A架开机"
    if prev_ajia5 == -1 and curr_ajia5 >= 0:
        df.loc[i, "status"] = "A架开机"

    # A架关机条件：当前 Ajia-3_v == -1，且前一时刻 Ajia-3_v >= 0
    if curr_ajia3 == -1 and prev_ajia3 >= 0:
        df.loc[i, "status"] = "A架关机"
    if curr_ajia5 == -1 and prev_ajia5 >= 0:
        df.loc[i, "status"] = "A架关机"

    # 有电流条件：前一时刻有一个或全部为0，下一刻均不为0
    if (prev_ajia3 <= 0 or prev_ajia5 <= 0) and (curr_ajia3 > 0 and curr_ajia5 > 0):
        df.loc[i, "check_current_presence"] = "有电流"
    # 无电流条件：前一时刻均不为0，下一刻有一个或全部为0
    elif prev_ajia3 > 0 and prev_ajia5 > 0 and (curr_ajia3 <= 0 or curr_ajia5 <= 0):
        df.loc[i, "check_current_presence"] = "无电流"

logger.success("A架开关机和有无电流判定完成")

2025-02-21 23:13:58.197 [SPECIAL] 开始判定A架开关机和有无电流
2025-02-21 23:14:01.089 [SUCCESS] A架开关机和有无电流判定完成


In [4]:
# 处理A架角度数据
def compute_angle_range(row):
    """
    计算角度范围
    """
    if row["Ajia-0_v"] == "error" or row["Ajia-1_v"] == "error":
        return "error"
    return abs(float(row["Ajia-0_v"]) - float(row["Ajia-1_v"]))

logger.special("开始处理A架角度范围")
df["angle_range"] = df.apply(compute_angle_range, axis=1)
logger.success("处理完成")

2025-02-21 23:14:01.102 [SPECIAL] 开始处理A架角度范围
2025-02-21 23:14:01.572 [SUCCESS] 处理完成


In [5]:
# 根据开关机事件，将A架数据分为若干段
logger.special("根据开关机事件，将A架数据分为若干段")
start_time = None
segments = []

for index, row in df.iterrows():
    if row["status"] == "A架开机":
        start_time = row["csvTime"]
    elif row["status"] == "A架关机" and start_time is not None:
        end_time = row["csvTime"]
        segments.append((start_time, end_time))
        start_time = None
        
logger.success("共分为%d段" % len(segments))
for i, (start_time, end_time) in enumerate(segments):
    logger.info(f"第{i+1}段：{start_time} - {end_time}")

2025-02-21 23:14:01.585 [SPECIAL] 根据开关机事件，将A架数据分为若干段
2025-02-21 23:14:06.051 [SUCCESS] 共分为95段
2025-02-21 23:14:06.051 [INFO] 第1段：2024-05-17 08:14:50 - 2024-05-17 10:15:50
2025-02-21 23:14:06.051 [INFO] 第2段：2024-05-17 19:00:50 - 2024-05-17 19:23:50
2025-02-21 23:14:06.051 [INFO] 第3段：2024-05-18 08:00:50 - 2024-05-18 09:12:50
2025-02-21 23:14:06.051 [INFO] 第4段：2024-05-18 09:33:50 - 2024-05-18 17:12:50
2025-02-21 23:14:06.051 [INFO] 第5段：2024-05-19 07:30:50 - 2024-05-19 15:39:50
2025-02-21 23:14:06.051 [INFO] 第6段：2024-05-19 15:41:50 - 2024-05-19 17:24:50
2025-02-21 23:14:06.052 [INFO] 第7段：2024-05-20 07:00:50 - 2024-05-20 07:35:50
2025-02-21 23:14:06.052 [INFO] 第8段：2024-05-20 12:52:50 - 2024-05-20 17:00:50
2025-02-21 23:14:06.052 [INFO] 第9段：2024-05-21 11:00:49 - 2024-05-21 12:23:49
2025-02-21 23:14:06.052 [INFO] 第10段：2024-05-21 15:00:49 - 2024-05-21 18:00:49
2025-02-21 23:14:06.052 [INFO] 第11段：2024-05-21 18:14:49 - 2024-05-21 18:20:49
2025-02-21 23:14:06.052 [INFO] 第12段：2024-05-22 07:51:49 -

In [6]:
# 让LLM预测不好判断的动作
XIAFANG = """你非常细心，通过仔细给定观察序列数据电流变化，尽可能正确返回三个值。
例如：
电流变化序列数据：
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.6478, 56.5133, 60.8637, 56.3751, 56.3777, 56.3601, 61.1564, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 422.499, 56.2896, 66.3951, 60.8928, 57.7813, 56.3871, 66.3077, 62.5263, 56.3937, 58.0826, 90.0969, 87.5592, 83.9934, 56.5033, 59.3441, 58.0018, 56.3027, 56.2845, 56.3666, 101.763, 96.6118, 56.3492, 59.2629, 57.0112, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]
在这里面寻找三个值返回结果：
[90.0969,56.5033,101.763]
返回依据解释：
第一个值为 最后一段非零数据的第一次达到峰值，同时满足必须为80以上；
第二个值为 要小于60，峰值回落至小于60的值,并且要在第二个值后面，在第三个值前面；
第三个值为 重新达到峰值，同时满足必须为90以上。
三个值均不考虑大于200异常数据


现有一组新的电流变化序列数据：
<<L>>
请参照样例和解释依据，新的电流变化序列数据可能会有一些噪声，你自己仔细思考判断，思考完成后，不需要返回思考过程，以列表形式返回三个值,回答中只有列表。。"""

HUISHOU = """你非常细心，通过仔细给定观察序列数据电流变化，尽可能正确返回三个值。
例如：
电流变化序列数据：
[0.0, 0.0, 0.0, 0.0, 0.0, 57.0048, 56.8545, 61.9802, 56.8646, 56.8705, 56.777, 68.3751, 56.5526, 56.6556, 63.1736, 68.4542, 78.2151, 86.3214, 82.7017, 58.9111, 56.632, 56.9142, 56.6583, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.2542, 56.2177, 56.1263, 56.2697, 56.102, 59.5568, 57.5703, 57.6415, 56.9307, 57.0531, 56.9337, 58.582, 58.0159, 104.238, 96.6301, 97.1496, 56.5543, 63.426, 57.6552, 56.6086, 56.6611, 56.5601, 56.6476, 56.68, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]
在这里面寻找三个值返回结果：
[86.3214,104.238,56.5543]
返回依据解释：
第一个值为 非最后一段非零数据的峰值，同时满足必须为80以上；
第二个值为后面一段非零数据的峰值，一般要为90以上；
第三个值 要小于60，峰值回落至小于60的值,并且要在第二个值后面。
三个值均不考虑大于200异常数据


现有一组新的电流变化序列数据：
<<L>>
请参照样例和解释依据，新的电流变化序列数据可能会有一些噪声，你自己仔细思考判断，思考完成后，不需要返回思考过程，以列表形式返回三个值,,回答中只有列表。。"""


def predict_sequence_by_llm(L_sequence, oper):
    from api import get_completion

    if oper == 0:
        context_text = XIAFANG
    else:
        context_text = HUISHOU
    prompt = context_text.replace("<<L>>", L_sequence)
    logger.info("【LLM请求】：%s" % prompt)
    messages = [{"role": "user", "content": prompt}]
    response = get_completion(messages)
    return str(response.choices[0].message.content)


def get_result(L_sequence, oper):
    try:
        input_string = predict_sequence_by_llm(L_sequence=L_sequence, oper=oper)
        logger.info(f"【LLM返回】：{input_string}")
        result_list = json.loads(input_string)

        if len(result_list) == 3:
            a = result_list[0]
            b = result_list[1]
            c = result_list[2]
            return a, b, c
    except:
        try:
            input_string = predict_sequence_by_llm(L_sequence=L_sequence)
            result_list = json.loads(input_string)
            if len(result_list) == 3:
                a = result_list[0]
                b = result_list[1]
                c = result_list[2]
                return a, b, c
        except:
            return -100, -100, -100

In [7]:
# 判断A架关键动作的辅助函数
def extract_daily_power_on_times(df):
    """
    从CSV文件中提取一天内有两次开机的第一次和第二次开机时间。

    参数:
    file_path (str): CSV文件的路径，包含 'csvTime' 和 'status' 列。

    返回:
    first_start_times (list): 一天内有两次开机的第一次开机时间列表。
    second_start_times (list): 一天内有两次开机的第二次开机时间列表。
    """
    # 读取CSV文件
    df = df

    # 将 csvTime 转换为 datetime 类型
    df["csvTime"] = pd.to_datetime(df["csvTime"])

    # 按天分组
    df["date"] = df["csvTime"].dt.date

    # 初始化一个字典来存储每天的开机关机时间段
    daily_segments = {}

    # 遍历每一天的数据
    for date, group in df.groupby("date"):
        segments = []
        start_time = None

        # 遍历每一天的记录
        for index, row in group.iterrows():
            if row["status"] == "A架开机":
                start_time = row["csvTime"]
            elif row["status"] == "A架关机" and start_time is not None:
                end_time = row["csvTime"]
                segments.append((start_time, end_time))
                start_time = None

        # 将每天的开机关机时间段存入字典
        daily_segments[date] = segments

    # 统计每天的开机关机次数
    daily_counts = {date: len(segments) for date, segments in daily_segments.items()}

    # 筛选出一天内有两次开机关机的情况
    two_times_days = [date for date, count in daily_counts.items() if count == 2]

    # 初始化两个列表来存储第一次和第二次的开机时间
    first_start_times = []
    second_start_times = []

    # 遍历这些日期，提取第一次和第二次的开机时间
    for date in two_times_days:
        segments = daily_segments[date]
        first_start_times.append(segments[0][0])  # 第一次开机时间
        second_start_times.append(segments[1][0])  # 第二次开机时间

    return first_start_times, second_start_times


def find_peaks(data1):
    # 数据预处理
    data = [50 if 50 <= num <= 68 else num for num in data1]

    # 找到峰值
    peaks = []
    for i in range(1, len(data) - 1):  # 从第二个元素遍历到倒数第二个元素
        if data[i] > data[i - 1] and data[i] > data[i + 1]:  # 判断是否为峰值
            peaks.append(data[i])  # 只记录峰值值
    peaks = [peak for peak in peaks if peak > 80]
    # 返回峰值格式和具体的峰值
    return len(peaks), peaks


def find_first_increasing_value(data):
    """
    找到列表中第一个从稳定值（68以下）开始增加的值。

    参数:
    data (list): 输入的数值列表。

    返回:
    tuple: 第一个大于68的值及其索引。如果未找到，返回 (None, None)。
    """
    # 将介于50到68之间的值替换为50
    processed_data = [50 if 50 <= num <= 68 else num for num in data]

    # 找到第一个大于68的值及其索引
    for index, value in enumerate(processed_data):
        if value > 68 and value < 300:
            return value
    # 如果未找到，返回 (None, None)
    return 50


def find_stable_value(data1, data2, peak1, peak2):
    """
    找到两个峰值之间的数据中，回落到稳定值的第一个值。
    假设稳定值在 50 到 60 之间。

    参数:
    data (list): 数据列表
    peak1 (float): 第一个峰值
    peak2 (float): 第二个峰值

    返回:
    float or None: 稳定值，如果未找到则返回 None
    """
    # 找到峰值之间的数据
    try:
        start_index = data1.index(peak1)
        end_index = data1.index(peak2)
    except ValueError:
        # 如果峰值不在列表中，返回 None
        return None

    between_peaks1 = data1[start_index : end_index + 1]
    between_peaks2 = data2[start_index : end_index + 1]

    # 找到回落到稳定值的第一个值（假设稳定值在 50 到 60 之间）
    for index, value in enumerate(between_peaks1):
        if 50 <= value <= 60 and 50 <= between_peaks2[index] <= 60:
            return value

    # 如果未找到稳定值，返回 None
    return None


def find_first_stable_after_peak(data, peak, stable_min=50, stable_max=60):
    """
    从峰值到列表末尾的数据中，找到第一个回落到稳定值的值。

    参数:
    data (list): 数据列表
    peak (float): 峰值
    stable_min (float): 稳定值的最小值
    stable_max (float): 稳定值的最大值

    返回:
    float or None: 稳定值，如果未找到则返回 None
    """
    try:
        # 找到峰值的索引
        start_index = data.index(peak)
    except ValueError:
        # 如果峰值不在列表中，返回 None
        return None

    # 切片获取从峰值到列表末尾的数据
    after_peak = data[start_index:]

    # 找到回落到稳定值的第一个值
    for value in after_peak:
        if stable_min <= value <= stable_max:
            return value

    # 如果未找到稳定值，返回 None
    return None


def extract_events(df, segment):
    """
    从数据中提取事件对
    """
    start, end = segment
    # start = pd.to_datetime(start)
    # end = pd.to_datetime(end)
    logger.info(f"【提取事件对】开机时间: {start}, 关机时间: {end}")
    events = df[
        (df["csvTime"] >= start)
        & (df["csvTime"] <= end)
        & (df["check_current_presence"].isin(["有电流", "无电流"]))
    ]
    logger.info(f"【提取事件对】事件数量: {events.shape[0]}")
    L3 = []
    # 检查事件数量是否为偶数
    if events.shape[0] >= 2 and events.shape[0] % 2 == 0:
        # 遍历所有偶数索引的事件对
        for i in range(0, events.shape[0], 2):
            event_start = events.iloc[i]
            event_end = events.iloc[i + 1]
            # 确保第一个事件是“有电流”，第二个事件是“无电流”
            if (
                event_start["check_current_presence"] == "有电流"
                and event_end["check_current_presence"] == "无电流"
            ):
                start_event_time = event_start["csvTime"]
                end_event_time = event_end["csvTime"]

                # 提取两个事件之间的数据
                between_events = df[
                    (df["csvTime"] >= start_event_time)
                    & (df["csvTime"] <= end_event_time)
                ]
                data1 = list(between_events["Ajia-5_v"])
                logger.info(f"【提取事件对】事件对 ({i}, {i + 1}) 之间的数据: {data1}")

                # 调用 find_peaks 函数（假设已定义）
                len_peaks, peak_L = find_peaks(data1)
                logger.info(
                    f"【提取事件对】事件对 ({i}, {i + 1}) 之间的峰值数量: {len_peaks}，峰值为{peak_L}"
                )
                L3.append(len_peaks)
    return L3

In [ ]:
# 判定A架的关键动作
# 提取每个区段内的“通电流”和“关电流”事件
for segment in segments:
    start, end = segment
    logger.success(f"【开始处理时间段】开机时间: {start}, 关机时间: {end}")
    L4 = extract_events(df, segment=segment)
    logger.info(f"【处理时间段】区间类型：{L4}")
    segment_data = df[(df["csvTime"] >= start) & (df["csvTime"] <= end)]
    logger.info(f"【处理时间段】区间数据：{list(segment_data['Ajia-5_v'])}")
    if L4 == [0, 2] or L4 == [0, 0, 2]:
        events = df[
            (df["csvTime"] >= start)
            & (df["csvTime"] <= end)
            & (df["check_current_presence"].isin(["有电流", "无电流"]))
        ]
        if events.shape[0] % 2 == 0:
            if (
                events.iloc[0]["check_current_presence"] == "有电流"
                and events.iloc[1]["check_current_presence"] == "无电流"
            ):
                start_event_time = events.iloc[0]["csvTime"]
                end_event_time = events.iloc[1]["csvTime"]
                between_events = df[
                    (df["csvTime"] >= start_event_time)
                    & (df["csvTime"] <= end_event_time)
                ]
                data1 = list(between_events["Ajia-5_v"])

                len_peaks, peak_L = find_peaks(data1)

                if len_peaks == 0:
                    if (
                        events.iloc[2]["check_current_presence"] == "有电流"
                        and events.iloc[3]["check_current_presence"] == "无电流"
                    ):
                        if L4 == [0, 2]:
                            start_event_time = events.iloc[2]["csvTime"]
                            end_event_time = events.iloc[3]["csvTime"]
                        elif L4 == [0, 0, 2]:
                            start_event_time = events.iloc[4]["csvTime"]
                            end_event_time = events.iloc[5]["csvTime"]
                        between_events = df[
                            (df["csvTime"] >= start_event_time)
                            & (df["csvTime"] <= end_event_time)
                        ]
                        data1 = list(between_events["Ajia-5_v"])
                        data2 = list(between_events["Ajia-3_v"])
                        len_peaks, peak_L = find_peaks(data1)
                        if len_peaks == 2:
                            value_11 = find_first_increasing_value(data1)
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "征服者起吊"

                            value_11 = find_stable_value(
                                data1, data2, peak_L[0], peak_L[1]
                            )
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "缆绳解除"
                            previous_indices = [idx - 1 for idx in indices if idx > 0]
                            df.loc[previous_indices, "status"] = "征服者入水"

                            # 找到 between_events 中 Ajia-5_v 等于 target_value 的索引
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == peak_L[1]
                            ].tolist()
                            df.loc[indices, "status"] = "A架摆回"
    if L4 == [2]:
        events = df[
            (df["csvTime"] >= start)
            & (df["csvTime"] <= end)
            & (df["check_current_presence"].isin(["有电流", "无电流"]))
        ]
        if events.shape[0] % 2 == 0:
            if (
                events.iloc[0]["check_current_presence"] == "有电流"
                and events.iloc[1]["check_current_presence"] == "无电流"
            ):
                start_event_time = events.iloc[0]["csvTime"]
                end_event_time = events.iloc[1]["csvTime"]
                between_events = df[
                    (df["csvTime"] >= start_event_time)
                    & (df["csvTime"] <= end_event_time)
                ]
                data1 = list(between_events["Ajia-5_v"])

                len_peaks, peak_L = find_peaks(data1)

                if len_peaks == 2:
                    if (
                        events.iloc[0]["check_current_presence"] == "有电流"
                        and events.iloc[1]["check_current_presence"] == "无电流"
                    ):
                        start_event_time = events.iloc[0]["csvTime"]
                        end_event_time = events.iloc[1]["csvTime"]
                        between_events = df[
                            (df["csvTime"] >= start_event_time)
                            & (df["csvTime"] <= end_event_time)
                        ]
                        data1 = list(between_events["Ajia-5_v"])
                        data2 = list(between_events["Ajia-3_v"])
                        len_peaks, peak_L = find_peaks(data1)
                        if len_peaks == 2:
                            value_11 = find_first_increasing_value(data1)
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "征服者起吊"

                            value_11 = find_stable_value(
                                data1, data2, peak_L[0], peak_L[1]
                            )
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "缆绳解除"
                            previous_indices = [idx - 1 for idx in indices if idx > 0]
                            df.loc[previous_indices, "status"] = "征服者入水"

                            # 找到 between_events 中 Ajia-5_v 等于 target_value 的索引
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == peak_L[1]
                            ].tolist()
                            df.loc[indices, "status"] = "A架摆回"
    elif L4 == [0, 3]:
        events = df[
            (df["csvTime"] >= start)
            & (df["csvTime"] <= end)
            & (df["check_current_presence"].isin(["有电流", "无电流"]))
        ]
        if events.shape[0] % 2 == 0:
            if (
                events.iloc[0]["check_current_presence"] == "有电流"
                and events.iloc[1]["check_current_presence"] == "无电流"
            ):
                start_event_time = events.iloc[0]["csvTime"]
                end_event_time = events.iloc[1]["csvTime"]
                between_events = df[
                    (df["csvTime"] >= start_event_time)
                    & (df["csvTime"] <= end_event_time)
                ]
                data1 = list(between_events["Ajia-5_v"])

                len_peaks, peak_L = find_peaks(data1)

                if len_peaks == 0:
                    if (
                        events.iloc[2]["check_current_presence"] == "有电流"
                        and events.iloc[3]["check_current_presence"] == "无电流"
                    ):
                        start_event_time = events.iloc[2]["csvTime"]
                        end_event_time = events.iloc[3]["csvTime"]
                        between_events = df[
                            (df["csvTime"] >= start_event_time)
                            & (df["csvTime"] <= end_event_time)
                        ]
                        data1 = list(between_events["Ajia-5_v"])
                        data2 = list(between_events["Ajia-3_v"])
                        print(data1)
                        len_peaks, peak_L = find_peaks(data1)
                        if len_peaks == 3:
                            value_11 = find_first_increasing_value(data1)
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "征服者起吊"

                            value_11 = find_stable_value(
                                data1, data2, peak_L[1], peak_L[2]
                            )
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "缆绳解除"
                            previous_indices = [idx - 1 for idx in indices if idx > 0]
                            df.loc[previous_indices, "status"] = "征服者入水"

                            # 找到 between_events 中 Ajia-5_v 等于 target_value 的索引
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == peak_L[2]
                            ].tolist()
                            df.loc[indices, "status"] = "A架摆回"
    elif L4 == [0, 1, 3]:
        events = df[
            (df["csvTime"] >= start)
            & (df["csvTime"] <= end)
            & (df["check_current_presence"].isin(["有电流", "无电流"]))
        ]
        if events.shape[0] % 2 == 0:
            if (
                events.iloc[0]["check_current_presence"] == "有电流"
                and events.iloc[1]["check_current_presence"] == "无电流"
            ):
                start_event_time = events.iloc[0]["csvTime"]
                end_event_time = events.iloc[1]["csvTime"]
                between_events = df[
                    (df["csvTime"] >= start_event_time)
                    & (df["csvTime"] <= end_event_time)
                ]
                data1 = list(between_events["Ajia-5_v"])

                len_peaks, peak_L = find_peaks(data1)

                if len_peaks == 0:
                    if (
                        events.iloc[4]["check_current_presence"] == "有电流"
                        and events.iloc[5]["check_current_presence"] == "无电流"
                    ):
                        start_event_time = events.iloc[4]["csvTime"]
                        end_event_time = events.iloc[5]["csvTime"]
                        between_events = df[
                            (df["csvTime"] >= start_event_time)
                            & (df["csvTime"] <= end_event_time)
                        ]
                        data1 = list(between_events["Ajia-5_v"])
                        data2 = list(between_events["Ajia-3_v"])

                        len_peaks, peak_L = find_peaks(data1)
                        if len_peaks == 3:
                            value_11 = find_first_increasing_value(data1)
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "征服者起吊"
                            value_11 = find_stable_value(
                                data1, data2, peak_L[1], peak_L[2]
                            )
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "缆绳解除"
                            previous_indices = [idx - 1 for idx in indices if idx > 0]
                            df.loc[previous_indices, "status"] = "征服者入水"

                            # 找到 between_events 中 Ajia-5_v 等于 target_value 的索引
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == peak_L[2]
                            ].tolist()
                            df.loc[indices, "status"] = "A架摆回"
    elif L4 == [1, 2] or L4 == [1, 1]:
        events = df[
            (df["csvTime"] >= start)
            & (df["csvTime"] <= end)
            & (df["check_current_presence"].isin(["有电流", "无电流"]))
        ]
        if events.shape[0] % 2 == 0:
            if (
                events.iloc[0]["check_current_presence"] == "有电流"
                and events.iloc[1]["check_current_presence"] == "无电流"
            ):
                start_event_time = events.iloc[0]["csvTime"]
                end_event_time = events.iloc[1]["csvTime"]
                between_events = df[
                    (df["csvTime"] >= start_event_time)
                    & (df["csvTime"] <= end_event_time)
                ]
                data1 = list(between_events["Ajia-5_v"])

                len_peaks, peak_L = find_peaks(data1)

                if len_peaks == 1:
                    value_11 = find_first_increasing_value(data1)
                    indices = between_events.index[
                        between_events["Ajia-5_v"] == peak_L[0]
                    ].tolist()
                    df.loc[indices, "status"] = "A架摆出"
                    if (
                        events.iloc[2]["check_current_presence"] == "有电流"
                        and events.iloc[3]["check_current_presence"] == "无电流"
                    ):
                        start_event_time = events.iloc[2]["csvTime"]
                        end_event_time = events.iloc[3]["csvTime"]
                        between_events = df[
                            (df["csvTime"] >= start_event_time)
                            & (df["csvTime"] <= end_event_time)
                        ]
                        data1 = list(between_events["Ajia-5_v"])

                        len_peaks, peak_L = find_peaks(data1)
                        max_value = max([x for x in data1 if x <= 200])

                        if len_peaks == 2:
                            # 找到 between_events 中 Ajia-5_v 等于 target_value 的索引
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == max_value
                            ].tolist()
                            df.loc[indices, "status"] = "征服者出水"
                            previous_indices = [idx - 1 for idx in indices if idx > 0]
                            df.loc[previous_indices, "status"] = "缆绳挂妥"

                            value_11 = find_first_stable_after_peak(data1, max_value)
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "征服者落座"
                        elif len_peaks == 1:
                            # 找到 between_events 中 Ajia-5_v 等于 target_value 的索引
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == max_value
                            ].tolist()
                            df.loc[indices, "status"] = "征服者出水"
                            previous_indices = [idx - 1 for idx in indices if idx > 0]
                            df.loc[previous_indices, "status"] = "缆绳挂妥"

                            value_11 = find_first_stable_after_peak(data1, max_value)
                            indices = between_events.index[
                                between_events["Ajia-5_v"] == value_11
                            ].tolist()
                            df.loc[indices, "status"] = "征服者落座"
    else:
        segment_data = segment_data.copy()
        segment_data.loc[:, "csvTime"] = pd.to_datetime(segment_data["csvTime"])
        # 获取第一个值
        first_value = segment_data["csvTime"].iloc[0]

        # 定义目标日期
        target_date = datetime(2024, 8, 19)  # 2024年8月19日   大模型预测测试
        is_target_date = first_value.date() == target_date.date()

        target_date_1 = datetime(2024, 8, 19)  # 2024年8月19日   大模型预测测试
        is_target_date_1 = first_value.date() == target_date.date()

        # 判断小时是否大于12点
        is_hour_greater_than_12 = first_value.hour > 17
        first_start_times, second_start_times = extract_daily_power_on_times(df=df)

        if (
            is_target_date and is_hour_greater_than_12
        ):  # 全部预测可以去掉is_target_date条件 或者根据问题传入
            segment_data["new_column"] = segment_data.apply(
                lambda row: (
                    row["Ajia-3_v"]
                    if row["Ajia-5_v"] == 0 and row["Ajia-3_v"] > 0
                    else row["Ajia-5_v"]
                ),
                axis=1,
            )
            print("----------------LLM预测的列表---------------------")
            print(str(list(segment_data["new_column"])))

            try:
                a, b, c = get_result(str(list(segment_data["new_column"])), 1)
            except Exception as e:
                print(f"An error occurred: {e}")
                a, b, c = -100, -100, -100
            print("----------------预测的值---------------------")
            print(a, b, c)

            indices = segment_data.index[segment_data["new_column"] == a].tolist()
            df.loc[indices, "status"] = "A架摆出"

            indices = segment_data.index[segment_data["new_column"] == b].tolist()
            df.loc[indices, "status"] = "征服者出水"
            previous_indices = [idx - 1 for idx in indices if idx > 0]
            df.loc[previous_indices, "status"] = "缆绳挂妥"

            indices = segment_data.index[segment_data["new_column"] == c].tolist()
            df.loc[indices, "status"] = "征服者落座"

        elif (
            first_value in first_start_times and 1 == 0
        ):  # 去掉1==0由LLM判断状态  默认不开启 给大家分享参考思路。
            segment_data["new_column"] = segment_data.apply(
                lambda row: (
                    row["Ajia-3_v"]
                    if row["Ajia-5_v"] == 0 and row["Ajia-3_v"] > 0
                    else row["Ajia-5_v"]
                ),
                axis=1,
            )
            print("----------------LLM预测的列表---------------------")
            print(str(list(segment_data["new_column"])))

            try:
                a, b, c = get_result(str(list(segment_data["new_column"])), 0)
            except Exception as e:
                print(f"An error occurred: {e}")
                a, b, c = -100, -100, -100
            print("----------------预测的值---------------------")
            print(a, b, c)

            indices = segment_data.index[segment_data["new_column"] == a].tolist()
            df.loc[indices, "status"] = "征服者起吊"

            indices = segment_data.index[segment_data["new_column"] == b].tolist()
            df.loc[indices, "status"] = "缆绳解除"
            previous_indices = [idx - 1 for idx in indices if idx > 0]
            df.loc[previous_indices, "status"] = "征服者入水"

            indices = segment_data.index[segment_data["new_column"] == c].tolist()
            df.loc[indices, "status"] = "A架摆回"

        elif first_value in second_start_times and 1 == 0:  # 去掉1==0由LLM判断状态
            segment_data["new_column"] = segment_data.apply(
                lambda row: (
                    row["Ajia-3_v"]
                    if row["Ajia-5_v"] == 0 and row["Ajia-3_v"] > 0
                    else row["Ajia-5_v"]
                ),
                axis=1,
            )
            print("----------------LLM预测的列表---------------------")
            print(str(list(segment_data["new_column"])))

            try:
                a, b, c = get_result(str(list(segment_data["new_column"])), 1)
            except Exception as e:
                print(f"An error occurred: {e}")
                a, b, c = -100, -100, -100
            print("----------------预测的值---------------------")
            print(a, b, c)

            indices = segment_data.index[segment_data["new_column"] == a].tolist()
            df.loc[indices, "status"] = "A架摆出"

            indices = segment_data.index[segment_data["new_column"] == b].tolist()
            df.loc[indices, "status"] = "征服者出水"
            previous_indices = [idx - 1 for idx in indices if idx > 0]
            df.loc[previous_indices, "status"] = "缆绳挂妥"

            indices = segment_data.index[segment_data["new_column"] == c].tolist()
            df.loc[indices, "status"] = "征服者落座"
    logger.success(f"【处理时间段完成】开机时间: {start}, 关机时间: {end}")

2025-02-21 23:14:06.262 [SUCCESS] 【开始处理时间段】开机时间: 2024-05-17 08:14:50, 关机时间: 2024-05-17 10:15:50
2025-02-21 23:14:06.262 [INFO] 【提取事件对】开机时间: 2024-05-17 08:14:50, 关机时间: 2024-05-17 10:15:50
2025-02-21 23:14:06.272 [INFO] 【提取事件对】事件数量: 4
2025-02-21 23:14:06.282 [INFO] 【提取事件对】事件对 (0, 1) 之间的数据: [56.4274, 56.3066, 0.0]
2025-02-21 23:14:06.282 [INFO] 【提取事件对】事件对 (0, 1) 之间的峰值数量: 0，峰值为[]
2025-02-21 23:14:06.290 [INFO] 【提取事件对】事件对 (2, 3) 之间的数据: [99.5829, 56.1955, 55.9012, 56.1437, 65.2289, 102.213, 98.5886, 110.858, 77.0694, 0.0]
2025-02-21 23:14:06.290 [INFO] 【提取事件对】事件对 (2, 3) 之间的峰值数量: 2，峰值为[102.213, 110.858]
2025-02-21 23:14:06.290 [INFO] 【处理时间段】区间类型：[0, 2]
2025-02-21 23:14:06.297 [INFO] 【处理时间段】区间数据：[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.9624, 59.8228, 56.7491, 56.5318, 56.7124, 64.3501, 79.9242, 56.3142, 58.7957, 56.4107, 56.6235, 80.7319, 56.4274, 56.3066, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 99.5829, 56.1955, 55.

In [ ]:
# 保存A架数据
logger.special("开始保存A架数据")
df = df.drop(columns=["date"])
# df = df.drop(columns=['check_current_presence'])
df.to_csv(os.path.join(output_path, "Ajia_plc_1.csv"), index=False)
logger.success("A架数据保存成功")

In [ ]:
# 判定ON DP和OFF DP
logger.special("开始判定ON DP和OFF DP")
df = pd.read_csv(os.path.join(output_path, "Port3_ksbg_9.csv"))
df["P3_33"] = pd.to_numeric(df["P3_33"], errors="coerce")
df["status"] = "False"

for i in range(1, df.shape[0]):
    # ON DP
    if df.loc[i - 1, "P3_33"] == 0 and df.loc[i, "P3_33"] > 0:
        df.loc[i, "status"] = "ON DP"
    # OFF DP
    if df.loc[i - 1, "P3_33"] > 0 and df.loc[i, "P3_33"] == 0:
        df.loc[i, "status"] = "OFF DP"

df.to_csv(os.path.join(output_path, "Port3_ksbg_9.csv"), index=False)
logger.success("ON DP和OFF DP数据保存成功")

In [ ]:
# 处理折臂吊车
from collections import Counter

logger.special("开始判定折臂吊车关键动作")
df = pd.read_csv(os.path.join(tmp_path, "device_13_11_meter_1311.csv"))
df["13-11-6_v"] = pd.to_numeric(df["13-11-6_v"], errors="coerce")
df["status"] = "False"
df["action"] = "False"


def sliding_window_5(arr):
    """滑动窗口大小为5的逻辑"""
    window_size = 5
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i : i + window_size]
        if (
            window[1] < 10
            and window[2] < 10
            and window[3] < 10
            and window[0] > 10
            and window[4] > 10
        ):
            # 将 window[0] 包装成列表进行赋值
            modified_arr[i + 1 : i + 4] = [window[0]] * 3
    return modified_arr


def sliding_window_4(arr):
    """滑动窗口大小为4的逻辑"""
    window_size = 4
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i : i + window_size]
        if window[1] < 10 and window[2] < 10 and window[0] > 10 and window[3] > 10:
            # 将 window[0] 包装成列表进行赋值
            modified_arr[i + 1 : i + 3] = [window[0]] * 2
    return modified_arr


def sliding_window_3(arr):
    """滑动窗口大小为3的逻辑"""
    window_size = 3
    modified_arr = arr.copy()
    for i in range(len(arr) - window_size + 1):
        window = arr[i : i + window_size]
        if window[1] < 10 and window[0] > 10 and window[2] > 10:
            # 直接赋值，因为只修改一个值
            modified_arr[i + 1] = window[0]
    return modified_arr


logger.info("【处理折臂吊车】开始应用滑动窗口逻辑")
df["13-11-6_v_new"] = sliding_window_5(df["13-11-6_v"].tolist())
df["13-11-6_v_new"] = sliding_window_4(df["13-11-6_v_new"].tolist())
df["13-11-6_v_new"] = sliding_window_3(df["13-11-6_v_new"].tolist())
logger.success("【处理折臂吊车】滑动窗口逻辑应用完成")

logger.info("【处理折臂吊车】开始判定折臂吊车的开机和关机事件")
for i in range(1, df.shape[0]):
    # 开机
    if df.iloc[i - 1]["13-11-6_v"] == 0 and df.iloc[i]["13-11-6_v"] > 0:
        df.at[df.index[i], "status"] = "折臂吊车开机"
    # 关机
    if df.iloc[i - 1]["13-11-6_v"] > 0 and df.iloc[i]["13-11-6_v"] == 0:
        df.at[df.index[i], "status"] = "折臂吊车关机"
    # 检测由待机进入工作和由工作进入待机的事件
    if df.iloc[i - 1]["13-11-6_v_new"] < 10 and df.iloc[i]["13-11-6_v_new"] > 10:
        df.at[df.index[i], "action"] = "由待机进入工作"
    if df.iloc[i - 1]["13-11-6_v_new"] > 10 and df.iloc[i]["13-11-6_v_new"] < 10:
        df.at[df.index[i], "action"] = "由工作进入待机"
logger.success("【处理折臂吊车】折臂吊车的开机和关机事件判定完成")

logger.info("【处理折臂吊车】根据折臂吊车的开机和关机事件划分时间段")
segments = []
start_time = None
for index, row in df.iterrows():
    if row["status"] == "折臂吊车开机":
        start_time = row["csvTime"]
    elif row["status"] == "折臂吊车关机" and start_time is not None:
        end_time = row["csvTime"]
        segments.append((start_time, end_time))
        start_time = None
logger.success("【处理折臂吊车】时间段划分完成")

def find_most_frequent_number(lst):
    """
    使用 Counter 统计每个数的出现次数\n
    找到出现次数最多的数（如果有多个，只返回第一个）
    """
    counter = Counter(lst)
    most_common_number = counter.most_common(1)[0][0]
    return most_common_number

df["csvTime"] = pd.to_datetime(df["csvTime"], errors="coerce")

for segment in segments:
    start, end = segment
    logger.info(f"【开始处理时间段】开始时间：{start}，结束时间：{end}")
    events = df[
        (df["csvTime"] >= start)
        & (df["csvTime"] <= end)
        & (df["action"].isin(["由待机进入工作", "由工作进入待机"]))
    ]
    segment_data = df[(df["csvTime"] >= start) & (df["csvTime"] <= end)]
    # 检查事件数量是否为偶数且等于6
    if events.shape[0] > 0 and events.iloc[0]["csvTime"] == start:
        events = events[2:]
        segment_data = segment_data[2:]
    if events.shape[0] == 8:
        # 计算时间差
        time_diffs = (
            events["csvTime"].iloc[1::2].values - events["csvTime"].iloc[::2].values
        ).astype("timedelta64[s]")
        # 找到最小时间差的位置
        min_idx = time_diffs.argmin() * 2
        # 直接 drop 对应索引
        events = events.drop(events.index[[min_idx, min_idx + 1]])
        
    logger.info(f"【处理时间段】事件数量: {events.shape[0]}")
    if events.shape[0] == 6:
        # 处理每一对事件
        for i in range(0, 6, 2):
            event_start = events.iloc[i]
            event_end = events.iloc[i + 1]

            if (
                event_start["action"] == "由待机进入工作"
                and event_end["action"] == "由工作进入待机"
            ):
                start_event_time = event_start["csvTime"]
                end_event_time = event_end["csvTime"]
                between_events = df[
                    (df["csvTime"] >= start_event_time)
                    & (df["csvTime"] <= end_event_time)
                ]
                data1 = list(between_events["13-11-6_v"])

                # 找到最后一个大于9的值
                last_value_above_9 = next((x for x in reversed(data1) if x > 9), None)

                if last_value_above_9 is not None:
                    all_indices = between_events.index[
                        between_events["13-11-6_v_new"] == last_value_above_9
                    ].tolist()
                    last_index = all_indices[-1] if all_indices else None

                    # 根据事件对的顺序更新status
                    if last_index is not None:
                        if i == 0:
                            df.loc[last_index, "status"] = "小艇检查完毕"
                        elif i == 2:
                            df.loc[last_index, "status"] = "小艇入水"
                        elif i == 4:
                            df.loc[last_index, "status"] = "小艇落座"
                else:
                    logger.info("列表中没有大于 9 的值")
    if events.shape[0] == 4:
        # 处理每一对事件
        for i in range(0, 4, 2):
            event_start = events.iloc[i]
            event_end = events.iloc[i + 1]
            if (
                event_start["action"] == "由待机进入工作"
                and event_end["action"] == "由工作进入待机"
            ):
                start_event_time = event_start["csvTime"]
                end_event_time = event_end["csvTime"]
                between_events = df[
                    (df["csvTime"] >= start_event_time)
                    & (df["csvTime"] <= end_event_time)
                ]
                data1 = list(between_events["13-11-6_v"])

                # 找到最后一个大于9的值
                last_value_above_9 = next((x for x in reversed(data1) if x > 9), None)

                if last_value_above_9 is not None:
                    all_indices = between_events.index[
                        between_events["13-11-6_v_new"] == last_value_above_9
                    ].tolist()
                    last_index = all_indices[-1] if all_indices else None

                    # 根据事件对的顺序更新status

                    if (
                        last_index is not None
                        and df.loc[last_index, "status"] == "False"
                    ):
                        if i == 0:
                            df.loc[last_index, "status"] = "小艇入水"
                        elif i == 2:
                            df.loc[last_index, "status"] = "小艇落座"
                else:
                    print("列表中没有大于 9 的值")
                # 保存结果
# df = df.drop(columns=['action'])
# df = df.drop(columns=['13-11-6_v_new'])
df.to_csv("data/device_13_11_meter_1311.csv", index=False)
logger.success("【处理折臂吊车】保存数据完成")

In [ ]:
# 移除临时文件夹
import shutil

if os.path.exists(tmp_path):
    shutil.rmtree(tmp_path)